In [4]:
import pandas as pd
from scipy.stats import weibull_min
import sys
import os
from datetime import datetime, timedelta
import json

In [7]:
# Add EAEET_v1 directory to path to import modules
current_dir = os.getcwd()

# If we're in the case study folder, go up one level
if os.path.basename(current_dir) == 'case study':
    project_root = os.path.dirname(current_dir)
else:
    # Search upward for EAEET_v1 folder
    project_root = current_dir
    found = False
    
    while project_root != os.path.dirname(project_root):  # Stop at filesystem root
        eaeet_v1_path = os.path.join(project_root, 'EAEET_v1')
        if os.path.exists(eaeet_v1_path) and os.path.isdir(eaeet_v1_path):
            found = True
            break
        project_root = os.path.dirname(project_root)
    
    if not found:
        # Fallback: assume EAEET_v1 is sibling to current directory
        project_root = os.path.dirname(current_dir)

# Add the PARENT directory of EAEET_v1 to sys.path, not EAEET_v1 itself
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added to path: {project_root}")

# Verify EAEET_v1 exists
eaeet_v1_path = os.path.join(project_root, 'EAEET_v1')
print(f"Looking for EAEET_v1 at: {eaeet_v1_path}")
print(f"EAEET_v1 exists: {os.path.exists(eaeet_v1_path)}")
print(f"Contents: {os.listdir(eaeet_v1_path) if os.path.exists(eaeet_v1_path) else 'N/A'}")

import EAEET_v1.simulations as sim 
import EAEET_v1.utils as utils

Looking for EAEET_v1 at: e:\emission_event\Emission-Estimation-Using-Emission-Events\EAEET_v1
EAEET_v1 exists: True
Contents: ['components.py', 'emission_event_converter.py', 'requirements.txt', 'sample_leak_rate.csv', 'simulations.py', 'utils.py', 'weather_permian.nc', '__pycache__']


#### Load plumes and sources downloaded from Carbon Mapper

In [8]:
# Load sources 
features = [] 
with open("CM_sources.json", 'r') as f: 
    data = json.load(f)
    feats = data["features"]
    for feat in feats: 
        if feat not in features: 
            features.append(feat) 
cm_sources = pd.DataFrame(features)

sources = []
for _,row in cm_sources.iterrows():
    plume_ids = row.properties.get("plume_ids") 
    plume_2024 = False  
    for plume_id in plume_ids: 
        if "2024" in plume_id:
            plume_2024 = True
    if plume_2024: 
        sources.append({
            "source_name": row.id, 
            "source_lon": row.geometry.get("coordinates")[0],
            "source_lat": row.geometry.get("coordinates")[1],
            "plume_count":row.properties.get("plume_count"),
            "scene_count":len(row.properties.get("observation_scenes_names")),
            "scene_names": row.properties.get("observation_scenes_names"),
            "plume_ids": row.properties.get("plume_ids"),
            "persistence": row.properties.get("persistence"),
            "published_at_max": row.properties.get("published_at_max"),
            "published_at_min": row.properties.get("published_at_min"),
            "timestamp_max": row.properties.get("timestamp_max"),
            "timestamp_min": row.properties.get("timestamp_min"),
            "source_rate": row.properties.get("emission_auto"),
            "source_rate_unc": row.properties.get("emission_uncertainty_auto"),
        })
        
source_df = pd.DataFrame(sources)

In [95]:
# load plumes
plume_df = pd.read_csv(r"CM_plumes.csv")

#### Join plume and source to create emission observations

In [96]:
emission_observations = [] 
for _,row in source_df.iterrows(): 
    plume_ids = row.plume_ids
    scene_names = row.scene_names 
    source_name = row.source_name
    detections_2024 = [] 
    plume_names_2024 = [] 
    for plume in plume_ids: 
        if "2024" in plume.split("t")[0] and plume not in plume_names_2024:
            plume_date = plume.split("-")[0][3:17]
            plume_date = datetime.strptime(plume_date, "%Y%m%dt%H%M%S")
            detections_2024.append(plume_date)
            plume_names_2024.append(plume)
    
    p = row.persistence # persistance
    lat = row.source_lat
    lon = row.source_lon
    ind = 0 
    for pid in plume_names_2024: 
        detected_plume = plume_df[plume_df.plume_id == pid]    
        emission_observations.append({
            "id": detected_plume["plume_id"].iloc[0],
            "rate": detected_plume["emission_auto"].iloc[0],
            "rate_uncertainty": detected_plume["emission_uncertainty_auto"].iloc[0],
            "source": source_name,
            "lon": lon,
            "lat": lat,
            "persistence": p,
            "observation_time": detections_2024[ind],
            "measurement_technology": detected_plume["instrument"].iloc[0],
        })
        ind += 1
    
    non_detects_2024 = [] 
    for scene_name in scene_names: 
        if scene_name:
            sn = scene_name.split(":")[-1]
            if sn not in non_detects_2024: 
                nd_time = datetime.strptime(sn[3:17], "%Y%m%dt%H%M%S")
                if nd_time not in detections_2024 and nd_time.year >= 2023:
                    non_detects_2024.append(nd_time)
    
    
    for nd in non_detects_2024:
        nd_str = nd.strftime("%Y-%m-%d %H:%M:%S")
        emission_observations.append({
            "id": source_name + '-' + nd_str,
            "rate": 0,
            "rate_uncertainty": 0,
            "source": source_name,
            "lon": lon,
            "lat": lat,
            "persistence": p,
            "observation_time": nd,
            "measurement_technology": "CM",
        })

In [97]:
EO = pd.DataFrame(emission_observations)

In [98]:
print(len(EO))
EO.head()

3993


,id,rate,rate_uncertainty,source,lon,lat,persistence,observation_time,measurement_technology
0,emi20240127t195915p13009-B,16605.41308,2120.261575,CH4_1B2_250m_-101.76426_31.85867?datetime=2024...,-101.76426,31.858674,0.285714,2024-01-27 19:59:01,emi
1,CH4_1B2_250m_-101.76426_31.85867?datetime=2024...,0.00000,0.000000,CH4_1B2_250m_-101.76426_31.85867?datetime=2024...,-101.76426,31.858674,0.285714,2024-10-24 17:37:00,CM
2,CH4_1B2_250m_-101.76426_31.85867?datetime=2024...,0.00000,0.000000,CH4_1B2_250m_-101.76426_31.85867?datetime=2024...,-101.76426,31.858674,0.285714,2024-10-09 17:46:05,CM
3,CH4_1B2_250m_-101.76426_31.85867?datetime=2024...,0.00000,0.000000,CH4_1B2_250m_-101.76426_31.85867?datetime=2024...,-101.76426,31.858674,0.285714,2025-01-01 17:50:00,CM
4,CH4_1B2_250m_-101.76426_31.85867?datetime=2024...,0.00000,0.000000,CH4_1B2_250m_-101.76426_31.85867?datetime=2024...,-101.76426,31.858674,0.285714,2024-11-30 18:03:03,CM


In [99]:
EO.to_csv("CM_emissions_observation.csv", index = False)

#### Convert plumes to emission events

In [100]:
event_df = pd.read_csv(r"CM_emission_events.csv")
event_df["startTime"] = pd.to_datetime(event_df["startTime"])
event_df["endTime"] = pd.to_datetime(event_df["endTime"])

In [101]:
event_df.head(2)

,id,rate,duration,quantity,sourceLocation,startTime,endTime,merged_from,uncertainties
0,23a2c94b-f939-408b-a510-9ba7aff08308,70.316401,4321.0000,3.038372e+05,CH4_1B1a_250m_-103.86877_31.98367?datetime=202...,2023-11-19 17:35:02,2024-05-17 18:35:02,GAO20240517t173526p0000-C,24.803941
1,80a8e068-5bdc-467e-964b-65d27c4a2098,5864.327475,5469.4825,3.207484e+07,CH4_1B2_250m_-101.26737_31.15282?datetime=2024...,2024-02-12 21:48:05,2024-09-27 19:17:02,emi20240731t181643p12010-A,854.840936


In [102]:
# expected duration distribution 
durations = list(event_df.duration) 
durations_clean = [x for x in durations if math.isfinite(x) and x>0]
dur_dist = list(utils.fitting_distribution(durations_clean,"weibull"))
# expected rate distribution 
clean_plume_df = event_df.dropna(subset = ["rate"])
rates_clean = list(clean_plume_df.rate)
rate_dist = list(utils.fitting_distribution(rates_clean,"weibull"))

In [103]:
uncertainty_percents = [] 
event_uncertainties = event_df.uncertainties
event_rates = event_df.rate

for ele in zip(event_rates,event_uncertainties): 
    uncertainty_percents.append(ele[1]/ele[0])

In [104]:
n_samples = 10000
rate_samples = [] 
for n in range(n_samples):
    sr = weibull_min.rvs(*rate_dist, size=1)[0]
    unc = np.random.choice(uncertainty_percents)
    if unc < 1: 
        sr = sr + sr * np.random.uniform(-unc, unc)
    else:
        sr = sr 
    rate_samples.append(sr)
duration_samples = list(weibull_min.rvs(*dur_dist, size=n_samples))

In [105]:
len(event_df.sourceLocation.unique())

491

In [106]:
len(event_df)

698

### Simulation 

In [107]:
rate_dist = {"UNKNOWN":rate_samples}
duration_dist = {"UNKNOWN":duration_samples}
MC = 300 
source_emissions = [] 
for source in event_df.sourceLocation.unique(): 
    source_event_df = event_df[event_df.sourceLocation == source].copy()
    pod = source_df[source_df.source_name == source].copy()
    prob_dist = {"UNKNOWN":pod.persistence.iloc[0]}

    total_pre_hours = 0 
    pre_emissions = 0
    pre_emissions_lower = 0 
    pre_emissions_upper = 0 
    for _,row in source_event_df.iterrows(): 
        est = row.startTime 
        if est < datetime(2024,1,1):
            est = datetime(2024,1,1)
        eed = row.endTime
        if eed > datetime(2025,1,1): 
            eed = datetime(2025,1,1)
    
    
        total_pre_hours += (eed - est).total_seconds()/3600
        pre = ((eed - est).total_seconds()/3600) * row.rate
        pre_emissions += pre
        pre_emissions_lower += ((eed - est).total_seconds()/3600) *  row.uncertainties
        pre_emissions_upper += ((eed - est).total_seconds()/3600) *  row.uncertainties

    if total_pre_hours >= 8784:
        print(f"no time to run extrapolation for UEs for {source}")
        total_emissions = pre_emissions
        total_emissions_upper = pre_emissions_upper
        total_emissions_lower = pre_emissions_lower
    else: 
        start_time = datetime(2024,1,1) + timedelta(hours = total_pre_hours)
        end_time = datetime(2024,12,31,23,59,59)
        sim_results = sim.extrapolation("by_site", 
                          prob_dist, rate_dist, duration_dist, start_time, end_time, MC)
        ue_emissions = np.median(sim_results.get("UNKNOWN"))
        ue_emissions_lower = ue_emissions - np.percentile(sim_results.get("UNKNOWN"),5)
        ue_emissions_upper = np.percentile(sim_results.get("UNKNOWN"),95) - ue_emissions
        total_emissions = pre_emissions + ue_emissions
        total_emissions_upper = (ue_emissions_upper**2 + pre_emissions_upper**2)**0.5 
        total_emissions_lower = (ue_emissions_lower**2 + pre_emissions_lower**2)**0.5 
    
    source_emissions.append({
        "source": source,
        "pre_emissions":pre_emissions,
        "pre_emissions_lower":pre_emissions_lower, 
        "pre_emissions_upper":pre_emissions_upper,
        "ue_emissions": ue_emissions,
        "ue_emissions_lower": ue_emissions_lower, 
        "ue_emissions_upper": ue_emissions_upper,
        "total_emissions": total_emissions,
        "total_emissions_lower":total_emissions_lower, 
        "total_emissions_upper": total_emissions_upper, 
    })

no time to run extrapolation for UEs for CH4_1B2_250m_-101.52275_32.51480?datetime=2024-01-01T00%3A00%3A00.000Z%2F2025-01-01T23%3A59%3A59.999Z&status=not_deleted
no time to run extrapolation for UEs for CH4_1B2_250m_-103.31693_32.02059?datetime=2024-01-01T00%3A00%3A00.000Z%2F2025-01-01T23%3A59%3A59.999Z&status=not_deleted
no time to run extrapolation for UEs for CH4_1B2_250m_-103.50422_32.06184?datetime=2024-01-01T00%3A00%3A00.000Z%2F2025-01-01T23%3A59%3A59.999Z&status=not_deleted
no time to run extrapolation for UEs for CH4_1B2_250m_-103.53327_32.05728?datetime=2024-01-01T00%3A00%3A00.000Z%2F2025-01-01T23%3A59%3A59.999Z&status=not_deleted
no time to run extrapolation for UEs for CH4_1B2_250m_-103.54998_32.10781?datetime=2024-01-01T00%3A00%3A00.000Z%2F2025-01-01T23%3A59%3A59.999Z&status=not_deleted
no time to run extrapolation for UEs for CH4_1B2_250m_-103.56127_32.16636?datetime=2024-01-01T00%3A00%3A00.000Z%2F2025-01-01T23%3A59%3A59.999Z&status=not_deleted
no time to run extrapolation

In [108]:
sdf = pd.DataFrame( source_emissions)
sdf.to_csv("total_emissions.csv")

In [109]:
all = 0 
for se in source_emissions:
    all += se.get('total_emissions')
print(all)

2289886581.9356403
